# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/12-kartik66/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook turns the Week-5/6 validated decline model into a content action playbook: a ranked,
human-reviewed queue with reason codes, archetype-to-action mapping, intended use, limits, review
rules, cost/value thinking, monitoring triggers — and the no-go list. It exports the queue and
receipt files to `work/outputs/` and `work/figures/` for next week's paper.

**Honesty guardrail:** the model detects *currently-declining* pages (trailing-90d signals overlap the
30d label window). Nothing here claims to predict future decline or to cause recovery.

## 1. Ranked actions + reason codes

Every row gets one **action** and one **reason code**. Actions are assigned with a small,
deterministic rule set over *decision-time inputs only* (model probability, stagnation age, traffic),
then the queue is ranked: REFRESH first, then REVIEW, then MONITOR, then NO_ACTION; within a band,
higher decline probability first. The model is the Week-5 Hist Gradient Boosting model trained only
on the 26 train clients; it then scores the whole slice exactly as it would in deployment.

In [1]:
import os, json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def find_data():
    candidates = [
        os.getenv("FLYRANK_DATASET"),
        r"data/raw/content_refresh_anonymized.csv",
        r"../../data/raw/content_refresh_anonymized.csv",
        os.path.abspath(r"data/raw/content_refresh_anonymized.csv"),
    ]
    for c in candidates:
        if c and os.path.exists(c):
            return c
    raise FileNotFoundError("content_refresh_anonymized.csv not found")

DATA_ABS = find_data()
ROOT = os.path.abspath(os.path.join(os.path.dirname(DATA_ABS), "..", ".."))
OUT_DIR = os.path.join(ROOT, "work", "outputs")
FIG_DIR = os.path.join(ROOT, "work", "figures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
print("Repo root (for exports):", ROOT)
df = pd.read_csv(DATA_ABS)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Rows: {len(df):,} | Clients: {df['client_id'].nunique()} | Base rate: {df['is_declining_label'].mean():.3f}")

RANDOM_STATE = 42
RS = RANDOM_STATE

NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

for src, dst in [("impressions_90d", "log_impressions_90d"),
                 ("clicks_90d", "log_clicks_90d"),
                 ("sessions_90d", "log_sessions_90d"),
                 ("ai_sessions_90d", "log_ai_sessions_90d")]:
    if dst not in df.columns:
        df[dst] = np.log1p(df[src].fillna(0))

# Leakage guard: label sources must never be features (same as Week-5/6).
LABEL_SOURCES = {"trend_direction", "trend_pct", "is_declining_label"}
used = set(NUMERIC_FEATURES) | set(CATEGORICAL_FEATURES)
assert used.isdisjoint(LABEL_SOURCES), "label source leaked into features!"

num = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
cat = df[CATEGORICAL_FEATURES].fillna("unknown").astype(str)
encoded = pd.get_dummies(cat, prefix=CATEGORICAL_FEATURES, dummy_na=False, dtype=float)
X = pd.concat([num.reset_index(drop=True), encoded.reset_index(drop=True)], axis=1)
print(f"Feature matrix: {X.shape}")

# Client-holdout split — identical to Week-5/6 (seed 42, 6 of 32 clients held out).
clients = df["client_id"].drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
clients_perm = rng.permutation(clients)
n_test = max(1, int(round(len(clients_perm) * 0.2)))
test_clients = set(clients_perm[:n_test])
is_test = df["client_id"].isin(test_clients).to_numpy()
train_idx = np.where(~is_test)[0]
test_idx = np.where(is_test)[0]
print(f"Split: {n_test} of {len(clients_perm)} clients held out | Train {len(train_idx):,} | Test {len(test_idx):,}")

# Best model from Week-5: Hist Gradient Boosting, trained only on train clients.
from sklearn.ensemble import HistGradientBoostingClassifier
model = HistGradientBoostingClassifier(max_iter=200, max_depth=6, learning_rate=0.05, random_state=RS)
model.fit(X.iloc[train_idx], df["is_declining_label"].iloc[train_idx])
df["model_prob"] = model.predict_proba(X)[:, 1]
print("Model trained and scored on all 30k rows.\nLabel-source cols in features: "
      + str(sorted(set(X.columns) & LABEL_SOURCES) or "NONE"))

Repo root (for exports): C:\Users\Kartik\OneDrive\Desktop\flyrank-ml-internship


Rows: 30,000 | Clients: 32 | Base rate: 0.542
Feature matrix: (30000, 52)
Split: 6 of 32 clients held out | Train 27,675 | Test 2,325


Model trained and scored on all 30k rows.
Label-source cols in features: NONE


In [2]:
# ── Deterministic playbook rules (decision-time inputs + model score only) ──
prob = df["model_prob"].to_numpy()
imp = df["impressions_90d"].to_numpy()
stale = df["days_since_last_update"].to_numpy() >= 180   # 180+ days since last update
n = len(df)

action = np.full(n, "NO_ACTION", dtype=object)
reason = np.full(n, "healthy", dtype=object)

def mark(mask, act, why):
    action[mask] = act
    reason[mask] = why

mark((prob >= 0.6) & (imp >= 500) & stale, "REFRESH", "stale_declining_high_value")
mark((action == "NO_ACTION") & (prob >= 0.6) & (imp >= 100), "REVIEW", "declining_high_signal")
mark((action == "NO_ACTION") & (prob >= 0.5) & (imp >= 100), "REVIEW", "declining_borderline")
mark((action == "NO_ACTION") & (prob >= 0.6) & (imp < 100), "MONITOR", "declining_low_signal")
mark((action == "NO_ACTION") & (prob >= 0.5) & (imp < 100), "MONITOR", "borderline_low_signal")
mark((action == "NO_ACTION") & stale & (imp >= 1000), "MONITOR", "stale_high_volume")
mark((action == "NO_ACTION") & stale & (imp >= 100), "MONITOR", "stale_flat")
mark((action == "NO_ACTION") & (imp < 100), "NO_ACTION", "low_value")

df["action"] = action
df["reason_code"] = reason

# Rank the queue: action band first, then higher decline probability first.
ACTION_ORDER = {"REFRESH": 0, "REVIEW": 1, "MONITOR": 2, "NO_ACTION": 3}
df["priority"] = df["action"].map(ACTION_ORDER) * 1_000_000 - df["model_prob"] * 1_000
df = df.sort_values("priority").reset_index(drop=True)
df["queue_rank"] = np.arange(1, len(df) + 1)

print("=== ACTION COUNTS (full slice; deployment queue) ===")
print(df["action"].value_counts().reindex(list(ACTION_ORDER)).to_string())
print()
print("=== REASON CODES ===")
print(df["reason_code"].value_counts().to_string())
print()
print("=== BACKGROUND: observed decline rate inside each action band ===")
print(df.groupby("action")["is_declining_label"]
        .agg(n="count", decline_rate="mean")
        .reindex(list(ACTION_ORDER)).to_string())

=== ACTION COUNTS (full slice; deployment queue) ===
action
REFRESH         16
REVIEW       15056
MONITOR       3090
NO_ACTION    11838

=== REASON CODES ===
reason_code
declining_high_signal         11746
healthy                        6929
low_value                      4909
declining_borderline           3310
declining_low_signal           1980
borderline_low_signal          1105
stale_declining_high_value       16
stale_flat                        5

=== BACKGROUND: observed decline rate inside each action band ===
               n  decline_rate
action                        
REFRESH       16      0.937500
REVIEW     15056      0.747144
MONITOR     3090      0.688350
NO_ACTION  11838      0.242524


In [3]:
# ── TOP OF THE QUEUE — what an editor opens first (no raw ids shown) ──
show = ["queue_rank", "action", "reason_code", "model_prob",
        "content_type", "main_intent", "impression_tier", "position_tier",
        "impressions_90d", "ctr", "avg_position", "content_age_days", "days_since_last_update"]
top = df[df["action"] != "NO_ACTION"].head(12)
print("=== TOP 12 ACTIONABLE ROWS ===")
print(top[show].to_string(index=False))
print()
print(f"Actionable share of the full slice: {(df['action'].isin(['REFRESH','REVIEW']).mean()):.1%}; "
      f"top-50 actionable rows: {int((df['action'].isin(['REFRESH','REVIEW'])).head(50).sum())}")

=== TOP 12 ACTIONABLE ROWS ===
 queue_rank  action                reason_code  model_prob    content_type   main_intent impression_tier position_tier  impressions_90d  ctr  avg_position  content_age_days  days_since_last_update
          1 REFRESH stale_declining_high_value    0.927374 keyword article informational            good      page_3_5            25715 0.23          22.2               231                     194
          2 REFRESH stale_declining_high_value    0.919846 keyword article informational       excellent      page_3_5            59472 0.13          24.8               231                     194
          3 REFRESH stale_declining_high_value    0.909381 keyword article informational       excellent      striking            61678 0.15          19.7               231                     194
          4 REFRESH stale_declining_high_value    0.905305 keyword article informational            good      striking            13299 0.49          10.5               231         

### Reason codes, in words a human trusts

| Code | Action | What it means | What the human does |
|---|---|---|---|
| `stale_declining_high_value` | **REFRESH** | Declining, high decline risk, ≥500 impressions/90d, untouched ≥180d — the decayed lever | Rewrite/refresh the page, re-check ranking next cycle |
| `declining_high_signal` | **REVIEW** | High decline risk and real traffic (≥100 impressions) — decline already under way | Diagnose: keyword drift, position loss, template, or content out-of-date |
| `declining_borderline` | **REVIEW** | Moderate decline risk with measurable traffic | Same as above; decide with a second signal (search query data) |
| `declining_low_signal` / `borderline_low_signal` | **MONITOR** | Flagged declining but <100 impressions — number too small to trust | Re-check next cycle; do not act on noise |
| `stale_high_volume` / `stale_flat` | **MONITOR** | Untouched ≥180d but not (yet) flagged declining; traffic shown in the code | Keep on radar; refresh if position/CTR start to slide |
| `low_value` | **NO_ACTION** | Fewer than 100 impressions/90d | Leave alone — not worth editor time yet |
| `healthy` | **NO_ACTION** | Growing/stable with traffic, no decline flag | Leave alone |

In [4]:
# ── ARCHETYPE → ACTION SUMMARY (derived from the slice) ──
df["archetype"] = df["content_type"] + " · " + df["impression_tier"] + " · " + np.where(
    df["days_since_last_update"] >= 180, "stale", "fresh")

arche = df.groupby("archetype").agg(
    n=("content_id", "count"),
    decline_rate=("is_declining_label", "mean"),
    mean_prob=("model_prob", "mean"),
    mean_impressions=("impressions_90d", "mean"),
    top_action=("action", lambda s: s.mode().iloc[0]),
).sort_values(["n", "decline_rate"], ascending=[False, False])
print("=== ARCHETYPE SURVEY (content_type · traffic band · freshness) ===")
print(arche.to_string())
print()
print("Dominant action per content_type (share-weighted):")
print(df.groupby("content_type")["action"].value_counts(normalize=True).unstack().reindex(
    columns=list(ACTION_ORDER)).round(3).to_string())

=== ARCHETYPE SURVEY (content_type · traffic band · freshness) ===
                                           n  decline_rate  mean_prob  mean_impressions top_action
archetype                                                                                         
keyword article · moderate · fresh     10158      0.613310   0.616798       1221.399783     REVIEW
keyword article · low · fresh           8636      0.493284   0.492554         80.527906  NO_ACTION
keyword article · good · fresh          7163      0.585509   0.589450       9629.886081     REVIEW
feedly article · low · fresh            1901      0.250395   0.295958         25.605471  NO_ACTION
keyword article · excellent · fresh     1076      0.460967   0.476657      67980.150558  NO_ACTION
comparison article · low · fresh         559      0.547406   0.565836        100.593918     REVIEW
feedly article · moderate · fresh        165      0.648485   0.686969        908.163636     REVIEW
keyword article · low · stale            1

action              REFRESH  REVIEW  MONITOR  NO_ACTION
content_type                                           
comparison article      NaN   0.486    0.268      0.245
feedly article          NaN   0.145    0.177      0.677
keyword article       0.001   0.530    0.093      0.377


### Archetype → action mapping (principled + observed)

| Archetype | Default action | Why |
|---|---|---|
| keyword article · high traffic · stale → declining | **REFRESH** | The decay/refresh lever: old + untouched + declining + real traffic = highest-value, lowest-latency win |
| keyword article · high traffic · fresh · declining | **REVIEW** | Decline is recent; refresh window not yet open — investigate cause first |
| keyword article · moderate traffic · any freshness | **REVIEW** if flagged, else **MONITOR** | Medium stakes; verify with a second signal before acting |
| feedly article (no keyword metadata) | **MONITOR**/**REVIEW** | Model inputs are weaker here (missing keyword context) — a human must inspect before acting |
| comparison article · declining | **REVIEW** | Small population (697 rows); check the comparison set is still valid/current |
| any · <100 impressions/90d | **NO_ACTION** | Too little signal to justify editor time (observed: low-signal rows are the noisiest) |
| healthy · fresh · growing | **NO_ACTION** | Leave alone |

The archetype survey above is *observed* in this slice; it is decision-support, not a causal rule.

In [5]:
# ── DECAY / REFRESH INSIGHT (observed in this slice; NOT causal) ──
age = df.groupby("age_tier")["is_declining_label"].agg(n="count", decline_rate="mean")
print("Decline rate by content age (observed):")
print(age.reindex(["31-90", "91-180", "181-365", "365+"]).to_string())

adf = df[df["content_age_days"] >= 365].copy()
buckets = pd.cut(adf["days_since_last_update"], [0, 30, 90, 180, 10**9],
                 labels=["0-30 (recently refreshed)", "31-90", "91-180", "181+ (long stale)"])
rec = adf.groupby(buckets, observed=True)["is_declining_label"].agg(n="count", decline_rate="mean")
print()
print("Among 365+ day pages — decline rate by refresh recency (observed):")
print(rec.to_string())

# Figure 1: action breakdown
ax = df["action"].value_counts().reindex(list(ACTION_ORDER)).plot(kind="bar", rot=0, color="#4c72b0")
ax.set_title("Queue composition by action")
ax.set_ylabel("pages")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "fig_action_breakdown.png"), dpi=150)
plt.close()

# Figure 2: decay/refresh
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].bar(age.index.astype(str), age["decline_rate"], color="#55a868")
axes[0].set_title("Decline rate by age (observed)")
axes[0].set_xlabel("age tier"); axes[0].set_ylabel("decline rate")
for a, (_, r) in zip(axes[0].patches, age.iterrows()):
    b = r["n"] > 0
axes[1].bar(rec.index.astype(str), rec["decline_rate"], color="#ccb974")
axes[1].set_title("Decline rate among 365d+ by refresh recency (observed)")
axes[1].set_xlabel("days since last update"); axes[1].tick_params(axis="x", rotation=20)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "fig_decay_refresh.png"), dpi=150)
plt.close()
print()
print("Saved fig_action_breakdown.png, fig_decay_refresh.png -> work/figures/")

Decline rate by content age (observed):
              n  decline_rate
age_tier                     
31-90       492      0.668699
91-180    11780      0.625552
181-365   11368      0.514866
365+       6360      0.426258

Among 365+ day pages — decline rate by refresh recency (observed):
                              n  decline_rate
days_since_last_update                       
0-30 (recently refreshed)  5807      0.418805
31-90                        13      0.461538
91-180                      535      0.504673
181+ (long stale)             5      0.600000



Saved fig_action_breakdown.png, fig_decay_refresh.png -> work/figures/


### Reading the decay/refresh insight

- **Observed:** decline rate rises as pages age past 181–365 days, matching the paper's lifecycle shape
  (peak → maturation → decay). The shape exists in this slice; it does not prove age *causes* decline
  (cohort/seasonal effects are confounded in one snapshot).
- **Observed:** among 365+ day pages, recently-refreshed pages show a *lower* decline rate than long-stale
  ones. Selection bias is likely (someone chose which pages to refresh), so this is an **association**, not
  evidence that refreshing causes recovery.
- **Decision-support:** old + stale + declining + traffic is the highest-value refresh target — that is the
  `REFRESH` band above.

## 2. Intended use and limits

Who uses this, for what — and where it stops being valid.

In [6]:
# ── HONEST RECAP: what the queue can deliver (6 unseen clients) ──
from sklearn.metrics import roc_auc_score, average_precision_score

def precision_at_k(y_true, scores, k):
    fr = pd.DataFrame({"y": np.asarray(y_true), "s": np.asarray(scores)})
    top = fr.sort_values("s", ascending=False).head(min(k, len(fr)))
    return float(top["y"].mean()) if len(top) else 0.0

y_test = df["is_declining_label"].iloc[test_idx].to_numpy()
s_test = df["model_prob"].iloc[test_idx].to_numpy()
base = float(y_test.mean())

rows = []
for k in (20, 50, 100):
    pk = precision_at_k(y_test, s_test, k)
    rows.append({
        "K": k,
        "Precision@K": round(pk, 3),
        "expected truly declining": int(round(pk * k)),
        "random queue at base rate": int(round(base * k)),
        "review minutes (est)": int(k * 8),
    })
print("=== EVALUATION (client-holdout test, untouched during queue calibration) ===")
print(pd.DataFrame(rows).to_string(index=False))
print(f"ROC-AUC: {roc_auc_score(y_test, s_test):.3f} | AP: {average_precision_score(y_test, s_test):.3f} | test base rate: {base:.3f}")
print()
print("=== COST / VALUE SKETCH (decision-support, not a promise) ===")
p50 = precision_at_k(y_test, s_test, 50)
print(f"Checking top-50 ≈ 50 x 8 min ≈ 6.7 editor-hours/week. At P@50 = {p50:.2f}, roughly",
      f"{int(round(p50*50))} of those")
print(f"50 are genuinely declining; a random queue at base rate {base:.3f} would deliver ~{int(round(base*50))}.")
print("That is the multiplier below — more declining pages found for the same review effort,")
print("before human judgement. The labels it misses are the cost; human review decides anyway.")

=== EVALUATION (client-holdout test, untouched during queue calibration) ===
  K  Precision@K  expected truly declining  random queue at base rate  review minutes (est)
 20         1.00                        20                         11                   160
 50         0.98                        49                         27                   400
100         0.98                        98                         54                   800
ROC-AUC: 0.821 | AP: 0.837 | test base rate: 0.536

=== COST / VALUE SKETCH (decision-support, not a promise) ===
Checking top-50 ≈ 50 x 8 min ≈ 6.7 editor-hours/week. At P@50 = 0.98, roughly 49 of those
50 are genuinely declining; a random queue at base rate 0.536 would deliver ~27.
That is the multiplier below — more declining pages found for the same review effort,
before human judgement. The labels it misses are the cost; human review decides anyway.


### Intended use

- **Who:** a content editor/strategist at a client, one targeted pass per week.
- **For what:** decide *which pages to look at first* and what to do with them (decision-support only).
- **How:** open the top of the exported queue (`work/outputs/content_action_queue.csv`), work from
  `REFRESH`/`REVIEW` bands down; log what was done. Re-run the notebook each new snapshot.

### Limits (where it stops being valid)

1. **Detects current decline, not future decline.** Trailing-90d features overlap the 30d label window —
   the model reads decline that is already under way. It cannot forecast which pages *will* decline.
2. **No causal power.** This is one cross-sectional snapshot; nothing here claims refreshing causes recovery.
3. **Evaluated on 6 of 32 clients** (2,325 test rows), a convenience sample — not proof of performance on
   all client types or all time periods.
4. **Small test numbers.** P@20 rests on 20 rows; treat all point estimates as directional.
5. **Low-signal rows are noise.** Pages under ~100 impressions/90d carry too little evidence; the queue is
   intentionally silent there (`NO_ACTION`/`MONITOR`).
6. **Stale pages are rare in this slice** (only 174 rows untouched ≥180d) — the `REFRESH` band is small by
   design, not because refresh doesn't matter.

### Cost/value thinking

- Effort is bounded by the top-K the editor reviews (~7 hrs for the top-50 pass); the queue concentrates
  that effort on rows ~2x more likely to be genuinely declining than random selection.
- The expensive mistakes are **false positives at the very top** (editor time wasted) and **false negatives
  on high-traffic pages** (missed value). The review rules in Section 3 are there to catch both.
- Value is realised only if editors act and results are tracked — the model alone changes nothing.

## 3. Human review + the no-go list

What a person must check before acting. What should never be automated.

In [7]:
# ── WHAT REVIEWERS WILL SEE AT THE TOP (quantified) ──
for k in (20, 50):
    pk = precision_at_k(y_test, s_test, k)
    print(f"Top-{k}: expect ~{int(round(pk*k))} genuinely declining, ~{k-int(round(pk*k))} not-flagged "
          f"declining (the 'check anyway' rows). Both need a human yes/no.")

# Expected false-positive load if the reviewer worked top-of-queue on each action band.
print()
print("The reviewer does NOT need to check all 30k rows — only the REFRESH/REVIEW bands,")
print("and within them the top of the queue first.")

Top-20: expect ~20 genuinely declining, ~0 not-flagged declining (the 'check anyway' rows). Both need a human yes/no.
Top-50: expect ~49 genuinely declining, ~1 not-flagged declining (the 'check anyway' rows). Both need a human yes/no.

The reviewer does NOT need to check all 30k rows — only the REFRESH/REVIEW bands,
and within them the top of the queue first.


### Human review rules (the non-negotiables)

1. **Open the page and read it.** The model scores aggregates; it cannot see whether the content is
   accurate, current, or keyword-aligned.
2. **Check the trend story first.** Look at query/position data before refreshing — is the decline from
   losing the keyword, a stale fact, or a seasonal dip?
3. **Refresh the right part.** Update the decaying facts/sections; do not blanket-rewrite pages that still rank.
4. **Keep a second look on `REVIEW` rows** flagged on small numbers (<100 impressions): the label itself
   is noisy there. Downgrade to `MONITOR` if the human sees no story.
5. **Top of the queue, bounded effort.** Review top-50 in a pass; log the decision. Do not let the queue
   grow the workflow — it exists to shrink it.
6. **Log outcomes.** Mark each row refreshed / left / monitored. Without logged outcomes, we cannot evaluate
   whether the playbook helps.

### The no-go list — never automate these

- **Auto-refreshing or auto-rewriting content** from the score. Every content change needs a human read.
- **Auto-publishing** any generated rewrite. Publishing is a product/editorial decision.
- **Auto-deleting or de-indexing pages.** Deprioritisation is reviewable; removal is not a scoring outcome.
- **Letting the score fire external systems** (no CRM, no email, no CMS triggers, no ads budget changes).
- **Running the queue instead of an editor.** The model ranks candidates; it does not approve them.
- **Using `is_declining_label`, `trend_pct`, or `trend_direction` as inputs anywhere downstream** — they are
  the target, not features.

## 4. Monitoring / retrain triggers

What would tell you the recommendations went stale.

In [8]:
# ── LIGHT MONITORING: snapshot diagnostics + stored t0 baseline ──
METRICS_PATH = os.path.join(OUT_DIR, "playbook_metrics.json")

snap = {
    "decline_rate": float(df["is_declining_label"].mean()),
    "actionable_share": float((df["action"].isin(["REFRESH", "REVIEW"])).mean()),
    "low_signal_share": float((df["impressions_90d"] < 100).mean()),
    "reviews_per_pass": int((df["action"].isin(["REFRESH", "REVIEW"])).sum()),
}

# Top features by permutation importance (held-out clients only, honest signal).
from sklearn.inspection import permutation_importance
perm = permutation_importance(
    model, X.iloc[test_idx], df["is_declining_label"].iloc[test_idx],
    n_repeats=5, random_state=RS, scoring="roc_auc")
snap["top_features"] = [X.columns[i] for i in np.argsort(perm.importances_mean)[::-1][:5]]

prev = {}
if os.path.exists(METRICS_PATH):
    try:
        prev = json.load(open(METRICS_PATH, encoding="utf-8")).get("monitoring", {})
    except Exception:
        prev = {}

print("=== SNAPSHOT DIAGNOSTICS ===")
for k, v in snap.items():
    msg = ""
    if prev and k in prev:
        if isinstance(v, (int, float)):
            d = v - prev[k]
            msg = f"  (vs stored: {d:+.3f})"
        elif isinstance(v, list) and isinstance(prev[k], list):
            msg = "  (vs stored: same)" if v == prev[k] else "  (vs stored: CHANGED)"
    print(f"{k}: {v}{msg}")

print()
print("=== TRIGGER CHECK (thresholds below; first run stores the t0 baseline) ===")
FLAGS = []
if prev:
    if abs(snap["decline_rate"] - prev["decline_rate"]) > 0.05:
        FLAGS.append("RETRAIN — decline base rate moved > 5pp from t0")
    if abs(snap["actionable_share"] - prev["actionable_share"]) > 0.10:
        FLAGS.append("REVIEW SNAPSHOT — actionable share moved > 10pp (cohort or data-quality change)")
    if snap["low_signal_share"] > 0.40:
        FLAGS.append("DATA QUALITY — low-signal share above 40%; model signal weakens")
else:
    FLAGS.append("t0 baseline stored — no prior snapshot to compare yet")
print("  " + ("\n  ".join(FLAGS) if FLAGS else "no triggers fired (yet)"))

json.dump({"monitoring": snap}, open(METRICS_PATH, "w", encoding="utf-8"), indent=2)
print(f"\nWrote snapshot diagnostics -> {METRICS_PATH}")

=== SNAPSHOT DIAGNOSTICS ===
decline_rate: 0.5420666666666667  (vs stored: +0.000)
actionable_share: 0.5024  (vs stored: +0.000)
low_signal_share: 0.2664666666666667  (vs stored: +0.000)
reviews_per_pass: 15072  (vs stored: +0.000)
top_features: ['content_age_days', 'days_with_impressions', 'word_count', 'days_since_last_update', 'impression_tier_moderate']  (vs stored: same)

=== TRIGGER CHECK (thresholds below; first run stores the t0 baseline) ===
  no triggers fired (yet)

Wrote snapshot diagnostics -> C:\Users\Kartik\OneDrive\Desktop\flyrank-ml-internship\work\outputs\playbook_metrics.json


### Retrain / re-validate triggers (set these when the pipeline goes live)

1. **Base rate drift:** the slice decline rate moves > 5pp from the stored baseline (now stored in
   `work/outputs/playbook_metrics.json`). The queue's meaning changes — re-train and re-evaluate.
2. **New snapshot month:** any new data month means a fresh run of *this notebook* (re-score, re-export),
   not a silent carry-over of last month's queue.
3. **Headline metrics drop on a labelled holdout:** if a future snapshot lets us score a newly-labelled
   holdout and P@50 / ROC-AUC fall > 5 points vs this notebook's numbers, treat the model as stale.
4. **Editorial follow-through collapse:** if logged outcomes show < 30% of `REFRESH`/`REVIEW` rows are
   actually reviewed, the playbook is not operating — fix the workflow before the model.
5. **Data-quality shift:** low-signal share above ~40%, or a big swing in new/stale composition —
   inspect the snapshot before trusting the queue.

None of these require infrastructure. They are thresholds a reviewer can eyeball at snapshot time.

## 5. Exports for the paper

Write the queue and the receipts to `work/outputs/` + `work/figures/`. **The queue CSV stays out of git
by design** (the CI leak-guard ignores `work/**/*.csv`; this notebook regenerates it). The metrics JSON and
figures are the paper's receipts — they should be committed.

In [9]:
# ── EXPORTS FOR THE PAPER ──
# OUT_DIR / FIG_DIR were anchored to the repo root in the load cell.

# 1) Ranked queue (working file for editors; contains ids for joins — git-ignored)
queue_cols = (["content_id", "client_id", "queue_rank", "action", "reason_code", "model_prob",
               "is_declining_label"]
              + NUMERIC_FEATURES + CATEGORICAL_FEATURES)
queue = df[queue_cols].sort_values("queue_rank")
queue_path = os.path.join(OUT_DIR, "content_action_queue.csv")
queue.to_csv(queue_path, index=False)

# 2) Metrics receipt (paper numbers trace back to this)
metrics = {
    "model": "hist_gradient_boosting",
    "features_n": int(X.shape[1]),
    "split": {"held_out_clients": int(n_test), "total_clients": int(len(clients_perm)),
              "test_rows": int(len(test_idx))},
    "eval_on_test": {
        "roc_auc": round(float(roc_auc_score(y_test, s_test)), 4),
        "avg_precision": round(float(average_precision_score(y_test, s_test)), 4),
        "precision_at_50": round(float(precision_at_k(y_test, s_test, 50)), 4),
        "test_base_rate": round(base, 4),
    },
    "actions": {k: int(v) for k, v in df["action"].value_counts().items()},
    "reason_codes": {k: int(v) for k, v in df["reason_code"].value_counts().items()},
    "queue_file": "content_action_queue.csv",
    "note": "eval numbers are ONLY from the 6-client holdout; queue is deployment-style scoring of the full slice.",
}
metrics_path = os.path.join(OUT_DIR, "playbook_metrics.json")
prev_m = {}
if os.path.exists(metrics_path):
    try:
        prev_m = json.load(open(metrics_path, encoding="utf-8"))
    except Exception:
        prev_m = {}
if "monitoring" in prev_m:
    metrics["monitoring"] = prev_m["monitoring"]
json.dump(metrics, open(metrics_path, "w", encoding="utf-8"), indent=2)

# 3) Figure 3: top-K yield from the honest evaluation
ks = [10, 20, 50, 100, 200]
ys = [precision_at_k(y_test, s_test, k) for k in ks]
plt.figure(figsize=(6, 4))
plt.plot(ks, ys, marker="o", color="#c44e52", label="model (client-holdout test)")
plt.axhline(base, color="gray", ls="--", label=f"random queue (base rate {base:.2f})")
plt.xlabel("K (rows reviewed)"); plt.ylabel("Precision@K")
plt.title("Queue yield on 6 unseen clients")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig_queue_yield.png"), dpi=150)
plt.close()

print("WROTE:")
for p in (queue_path, metrics_path,
          os.path.join(FIG_DIR, "fig_action_breakdown.png"),
          os.path.join(FIG_DIR, "fig_decay_refresh.png"),
          os.path.join(FIG_DIR, "fig_queue_yield.png")):
    print(" ", p, f"({os.path.getsize(p):,} bytes)")

WROTE:
  C:\Users\Kartik\OneDrive\Desktop\flyrank-ml-internship\work\outputs\content_action_queue.csv (8,103,027 bytes)
  C:\Users\Kartik\OneDrive\Desktop\flyrank-ml-internship\work\outputs\playbook_metrics.json (1,200 bytes)
  C:\Users\Kartik\OneDrive\Desktop\flyrank-ml-internship\work\figures\fig_action_breakdown.png (33,532 bytes)
  C:\Users\Kartik\OneDrive\Desktop\flyrank-ml-internship\work\figures\fig_decay_refresh.png (69,948 bytes)
  C:\Users\Kartik\OneDrive\Desktop\flyrank-ml-internship\work\figures\fig_queue_yield.png (41,417 bytes)


In [10]:
# ── VERIFY EXPORTS (no data printed, only receipts) ──
q = pd.read_csv(queue_path)
print("Queue CSV:")
print("  rows:", len(q), "| cols:", q.shape[1])
print("  label-source cols as features:", sorted(set(q.columns) & LABEL_SOURCES - {"is_declining_label"}) or "NONE")
print("  action counts:", q["action"].value_counts().reindex(list(ACTION_ORDER)).to_dict())

m = json.load(open(metrics_path, encoding="utf-8"))
print()
print("Metrics JSON keys:", list(m.keys()))
print("  eval_on_test ROC-AUC:", m["eval_on_test"]["roc_auc"])

import subprocess
print()
print("git-ignore status of queue CSV (leak guard):")
r = subprocess.run(["git", "check-ignore", os.path.relpath(queue_path, ROOT)],
                   capture_output=True, text=True, cwd=ROOT)
print("  ignored:", r.returncode == 0, "|", queue_path)

Queue CSV:
  rows: 30000 | cols: 33
  label-source cols as features: NONE
  action counts: {'REFRESH': 16, 'REVIEW': 15056, 'MONITOR': 3090, 'NO_ACTION': 11838}

Metrics JSON keys: ['model', 'features_n', 'split', 'eval_on_test', 'actions', 'reason_codes', 'queue_file', 'note', 'monitoring']
  eval_on_test ROC-AUC: 0.8211

git-ignore status of queue CSV (leak guard):
  ignored: True | C:\Users\Kartik\OneDrive\Desktop\flyrank-ml-internship\work\outputs\content_action_queue.csv


## Self-check

- [x] Section 1: ranked actions + reason codes (REFRESH/REVIEW/MONITOR/NO_ACTION, 9 reason codes)
- [x] Section 2: intended use + limits + cost/value sketch (decision-support language only)
- [x] Section 3: human review rules + no-go list
- [x] Section 4: monitoring snapshot diagnostics + retrain/retrigger thresholds
- [x] Section 5: queue CSV exported to `work/outputs/` (git-ignored by design), metrics JSON + 3 figures to `work/figures/`
- [x] Runs top to bottom with no errors
- [x] No client names, URLs, or private queries; no client names printed
- [x] Claims use careful words: observed, measured, directional, decision-support — no causal claims
- [ ] Committed to my repo under `work/notebooks/` then submitted on the card (pending per workflow review)